# Deliverable 2 — Generate, Grade, Audit, Benchmark

**Owners:** Pipeline lead (Steps 1–4), Security lead (Step 5), Schema lead (Step 6).

Prereqs: `data/gold_results.json` exists (Deliverable 1 notebook), and **your own** keys are in `.env`.

Generated SQL is cached in `results/generated_sql.json` — re-running this notebook will **not** call the APIs again for questions already done (one run per question per model, as the guide requires, and it protects your free quota).

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from src.paths import *
pd.set_option("display.max_colwidth", 120)

In [ ]:
questions = json.loads(QUESTIONS.read_text())
gold = {k: [tuple(r) for r in v] for k, v in json.loads((DATA / "gold_results.json").read_text()).items()}
schema_ddl = SCHEMA_DDL.read_text()
RESULTS.mkdir(exist_ok=True)
print(len(questions), "questions;", len(gold), "gold result sets")

## Step 3 — Run every question through both models (once each)

In [ ]:
import time
from src.llm import generate_sql, build_prompt, MODELS

cache_path = RESULTS / "generated_sql.json"
generated = json.loads(cache_path.read_text()) if cache_path.exists() else {}

for q in questions:
    for m in MODELS:
        key = f"{q['id']}|{m}"
        if key in generated:
            continue
        try:
            generated[key] = {"id": q["id"], "model_key": m, **generate_sql(m, schema_ddl, q["question"])}
            print(f"{key}: {generated[key]['sql'][:90]}")
        except Exception as e:
            print(f"{key}: FAILED — {e}  (fix and re-run; finished ones are cached)")
        cache_path.write_text(json.dumps(generated, indent=1))
        time.sleep(4)  # stay under free-tier requests-per-minute limits

print(f"{len(generated)}/{len(questions) * len(MODELS)} generations cached")

In [ ]:
# Save the exact prompt used for the report's methodology section
(RESULTS / "prompt_example.txt").write_text(build_prompt(schema_ddl, questions[0]["question"]))

## Step 4 — Grade correctness by comparing result sets

In [ ]:
from src.grading import run_query, results_match

rows = []
qmap = {q["id"]: q for q in questions}
for g in generated.values():
    q = qmap[g["id"]]
    pred, err = run_query(NORM_DB, g["sql"])
    rows.append({
        "id": g["id"], "model": g["model_key"], "sql": g["sql"],
        "correct": results_match(gold[g["id"]], pred, ordered=q.get("order_matters", False)),
        "error": err, "pred_rows": None if pred is None else len(pred), "gold_rows": len(gold[g["id"]]),
        "why_wrong": "",  # fill in by hand: wrong table / missed join / wrong aggregation / ...
    })
grades = pd.DataFrame(rows).sort_values(["id", "model"])
grades.to_csv(RESULTS / "correctness.csv", index=False)

summary = grades.groupby("model")["correct"].agg(["sum", "count"])
summary["score"] = summary["sum"].astype(str) + "/" + summary["count"].astype(str)
summary

In [ ]:
# Incorrect queries — annotate why_wrong in results/correctness.csv
grades[~grades["correct"]][["id", "model", "error", "pred_rows", "gold_rows", "sql"]]

## Step 5 — Security audit
The pre-screen below only **suggests** flags. The Security lead reviews every query by hand using `docs/security_checklist.md` and fills in the `reviewer_*` columns in `results/security_audit.csv` (open it in Excel/Sheets).

In [ ]:
from src.security import prescreen, CHECKS

audit_path = RESULTS / "security_audit.csv"
auto = pd.DataFrame([{"id": g["id"], "model": g["model_key"], "sql": g["sql"],
                      **{f"auto_{k}": v for k, v in prescreen(g["sql"]).items()}}
                     for g in generated.values()]).sort_values(["id", "model"])
if audit_path.exists():  # keep the reviewer's manual notes
    manual = pd.read_csv(audit_path)[["id", "model", "reviewer", "reviewer_verdict", "reviewer_notes"]]
    audit = auto.merge(manual, on=["id", "model"], how="left")
else:
    audit = auto.assign(reviewer="", reviewer_verdict="", reviewer_notes="")  # verdict: PASS / FLAG
audit.to_csv(audit_path, index=False)
flag_cols = [f"auto_{k}" for k in CHECKS]
audit[audit[flag_cols].any(axis=1)]

## Step 6 — Performance: 4 conditions (normalized/denormalized × no index/index)

In [ ]:
from src.benchmark import time_query, apply_indexes, drop_indexes, query_plan
import re

bench = json.loads((DATA / "benchmark_queries.json").read_text())

def index_names(stmts):
    return [re.search(r"INDEX\s+(?:IF NOT EXISTS\s+)?(\w+)", s, re.I).group(1) for s in stmts]

timings = []
for b in bench:
    for schema, db, sql, idx in [
        ("normalized", NORM_DB, b["normalized_sql"], b["normalized_indexes"]),
        ("denormalized", DENORM_DB, b["denormalized_sql"], b["denormalized_indexes"]),
    ]:
        names = index_names(idx)
        drop_indexes(db, names)
        timings.append({"id": b["id"], "schema": schema, "index": "no index",
                        "ms": time_query(db, sql) * 1000, "plan": query_plan(db, sql)})
        apply_indexes(db, idx)
        timings.append({"id": b["id"], "schema": schema, "index": "indexed",
                        "ms": time_query(db, sql) * 1000, "plan": query_plan(db, sql)})
        drop_indexes(db, names)  # leave DBs clean for the next run

perf = pd.DataFrame(timings)
perf.to_csv(RESULTS / "performance.csv", index=False)
perf.pivot_table(index="id", columns=["schema", "index"], values="ms").round(3)

In [ ]:
# The query plans show WHY the timings differ (SCAN vs SEARCH USING INDEX) — good for the report
for r in timings:
    print(f"--- {r['id']} {r['schema']} / {r['index']}: {r['ms']:.3f} ms\n{r['plan']}")

## Step 7 — Everything the report needs is now in `results/`. Draft it in `docs/report_template.md`.